In [0]:
output_customer_rfm_final_path = 's3://ims-ecom-stage/customer_rfm/'

In [0]:
customer_rfm_df = spark.sql("""
    SELECT
        c.customer_unique_id,
        DATEDIFF(DATE('2018-10-17'), DATE(MAX(o.order_purchase_timestamp))) AS recency,
        COUNT(distinct o.order_id) AS frequency,
        SUM(oi.price) AS monetary
    FROM ecommerce.staging.orders o
    JOIN ecommerce.staging.order_items oi
        ON o.order_id = oi.order_id
    JOIN ecommerce.staging.customers c
        ON o.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
""")

customer_rfm_df.createOrReplaceTempView("customer_rfm")

In [0]:
customer_rfm_final_df = spark.sql("""
SELECT
    customer_unique_id,
    recency,
    frequency,
    monetary,
    6 - NTILE(5) OVER (ORDER BY recency ASC) AS rm_recency_score,
    NTILE(5) OVER (ORDER BY monetary ASC) AS rm_monetary_score,
    CASE
        WHEN (6 - NTILE(5) OVER (ORDER BY recency ASC)) >= 4
         AND NTILE(5) OVER (ORDER BY monetary ASC) >= 4
            THEN 'High-Value Recent'

        WHEN (6 - NTILE(5) OVER (ORDER BY recency ASC)) >= 4
         AND NTILE(5) OVER (ORDER BY monetary ASC) <= 3
            THEN 'Recent Low-Spender'

        WHEN (6 - NTILE(5) OVER (ORDER BY recency ASC)) <= 3
         AND NTILE(5) OVER (ORDER BY monetary ASC) >= 4
            THEN 'At-Risk High-Value'

        ELSE 'Low-Value Inactive'
    END AS customer_segment
FROM customer_rfm
""")


In [0]:
customer_rfm_final_df.write \
    .format('delta') \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(output_customer_rfm_final_path)

In [0]:
  %sql
CREATE TABLE IF NOT EXISTS ecommerce.staging.customer_rfm
USING delta
LOCATION 's3://ims-ecom-stage/customer_rfm/';

In [0]:
%sql
select * from ecommerce.staging.customer_rfm